In [0]:
%run ../00-common/01-env-variables


In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df = spark.table(bronze_table)
display(circuits_df)

In [0]:
selected_circuits_df =circuits_df.select('circuitId' ,
                                         'circuitName' ,
                                          'lat',
                                           'lng' ,
                                           'locality', 
                                           'country',
                                           'ingestion_timestamp',
                                           'source_file')


In [0]:
display(selected_circuits_df)

In [0]:
renamed_circuits_df = selected_circuits_df.withColumnsRenamed(
    {"circuitId": "circuit_id",
   "circuitRef": "circuit_ref",
   "name": "circuit_name",
   "lat": "latitude",
   "lng": "longitude"
})

In [0]:
display(renamed_circuits_df)

In [0]:
from pyspark.sql import functions as F
non_null_circuts_df = renamed_circuits_df.filter(F.col('circuit_id').isNotNull())

In [0]:
display(non_null_circuts_df)

In [0]:
distinct_circuits = non_null_circuts_df.dropDuplicates(['circuit_id'])

In [0]:
display(distinct_circuits)

In [0]:
from pyspark.sql import functions as F

circuits_final_df = (
    distinct_circuits.withColumn('circuitName', F.initcap(F.col('circuitName')))
    .withColumn('locality', F.initcap(F.col("locality")))    
)

In [0]:
display(circuits_final_df)

In [0]:
(
    circuits_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)